In [16]:
# SageMath ≥ 9.x
# Lorentzian check
from sage.all import Polyhedron, ZZ, QQ, matrix, MatrixSpace, binomial, factorial
from itertools import product, combinations_with_replacement
import random
from sage.all import zero_matrix

# ============================================================
# Utilities: boundedness, fast integer-point counting (2D Pick)
# ============================================================

def scale_polyhedron(P, c):
    """
    Return c*P (Minkowski scaling) by scaling vertices.
    Works for bounded polyhedra (polytopes). c is a nonnegative integer.
    """
    if c == 0:
        return None  # we treat this as {0} at call sites
    return c*P

def is_bounded_poly(P):
    try:
        return P.is_compact()
    except AttributeError:
        return (len(P.rays_list()) == 0 and len(P.lines_list()) == 0)

def boundary_lattice_points_2d(P):
    B = ZZ(0)
    for e in P.faces(1):
        (x1,y1), (x2,y2) = [tuple(ZZ(c) for c in v.vector()) for v in e.vertices()]
        dx, dy = abs(x2-x1), abs(y2-y1)
        B += ZZ(dx).gcd(ZZ(dy))
    return B

def integral_points_count_fast(P, use_pick_2d=True, fallback_list_threshold=50000):
    if P.is_empty():
        return 0
    if use_pick_2d and P.ambient_dim() == 2:
        # Pick: |P∩Z^2| = Area + B/2 + 1  (exact for lattice polygons)
        A = P.volume()                     # exact rational
        B = boundary_lattice_points_2d(P)  # integer
        return ZZ((2*A + B + 2) // 2)
    try:
        return P.integral_points_count()
    except Exception:
        pts = P.integral_points()
        if len(pts) > fallback_list_threshold:
            raise RuntimeError("Too many integer points to enumerate safely.")
        return len(pts)

# ============================================================
# Minkowski-sum engine with scaling caches
# ============================================================

def _minkowski_sum_scaled(polys, mult, scaled_cache, sum_cache):
    key = tuple(mult)
    if key in sum_cache:
        return sum_cache[key]

    parts = []
    for i, c in enumerate(mult):
        if c == 0:
            continue
        if c == 1:
            parts.append(polys[i])
        else:
            skey = (i, c)
            Q = scaled_cache.get(skey)
            if Q is None:
                Q = scale_polyhedron(polys[i], c)
                scaled_cache[skey] = Q
            parts.append(Q)

    if not parts:
        sum_cache[key] = None  # represents {0}; caller handles count=1
        return None

    S = parts[0]
    for Q in parts[1:]:
        S = S.minkowski_sum(Q)
    sum_cache[key] = S
    return S


# ============================================================
# Grid of counts F(t) = |(Σ t_i P_i) ∩ Z^d| for 0<=t_i<=D
# ============================================================

def dimension_of_total(polys):
    S = polys[0]
    for Q in polys[1:]:
        S = S.minkowski_sum(Q)
    return S.dimension()

def precompute_counts_grid(polys, D, use_pick_2d=True):
    """
    counts[t] for t in [0..D]^k; counts[(0,...,0)] = 1 (the point {0}).
    """
    for P in polys:
        if not is_bounded_poly(P):
            raise ValueError("All inputs must be bounded polytopes.")
    k = len(polys)
    ranges = [range(D+1) for _ in range(k)]
    counts = {}
    scaled_cache, sum_cache = {}, {}

    for t in product(*ranges):
        if all(c == 0 for c in t):
            counts[t] = ZZ(1)
            continue
        S = _minkowski_sum_scaled(polys, t, scaled_cache, sum_cache)
        if not is_bounded_poly(S):
            raise ValueError("Minkowski sum became unbounded; check inputs.")
        counts[t] = ZZ(integral_points_count_fast(S, use_pick_2d=use_pick_2d))
    return counts

# ============================================================
# Binomial-basis coefficients c_alpha from the counts table
# c_alpha = Δ^alpha F(0) = Σ_{0<=s<=alpha} (-1)^{|α|-|s|} Π binom(α_i, s_i) F(s)
# ============================================================

def all_multiindices_upto(D, k):
    for a in product(*[range(D+1) for _ in range(k)]):
        yield a

def multi_le(a, b):
    return all(x <= y for x, y in zip(a, b))

def ell1(a):  # sum of coordinates
    return sum(a)

def multi_factorial(alpha):
    """
    alpha! = ∏_i (alpha_i!)
    """
    prod = 1
    for ai in alpha:
        prod *= factorial(ai)
    return prod

def coeffs_binomial_basis_from_counts(counts, D, k):
    """
    Returns dict c[alpha] for all alpha with |alpha| <= D.
    """
    c = {}
    for alpha in all_multiindices_upto(D, k):
        if ell1(alpha) > D:
            continue
        total = ZZ(0)
        # sum over s ≤ alpha
        ranges = [range(ai + 1) for ai in alpha]
        for s in product(*ranges):
            sign = -1 if ((ell1(alpha) - ell1(s)) % 2) else 1
            w = ZZ(1)
            for ai, si in zip(alpha, s):
                w *= binomial(ai, si)
            total += sign * w * counts[s]
        c[alpha] = total
    return c

# ============================================================
# Build homogeneous slice f_m(y) = Σ_{|α|=m} c_α y^α  over QQ
# ============================================================

def polynomial_slice(c, m, k, var_names=None, normalize=True):
    """
    Build the degree-m slice f_m(y) = sum_{|alpha|=m} (c_alpha / alpha!) y^alpha
    if normalize=True. Set normalize=False to get the raw (unnormalized) slice.
    """
    if var_names is None:
        var_names = [f"y{i+1}" for i in range(k)]
    R = PolynomialRing(QQ, k, names=var_names)
    y = R.gens()
    f = R.zero()
    for alpha, val in c.items():
        if sum(alpha) != m or val == 0:
            continue
        mon = R(1)
        for i, ai in enumerate(alpha):
            if ai:
                mon *= y[i]**ai
        if normalize:
            alpha_fact = 1
            for ai in alpha:
                alpha_fact *= factorial(ai)
            f += QQ(val) / QQ(alpha_fact) * mon
        else:
            f += QQ(val) * mon
    return f


# ============================================================
# M-convex support check (optional but useful)
# Support S ⊆ {α: |α|=m} is M-convex if:
# for any u,v∈S and any i with u_i>v_i, ∃ j with u_j<v_j such that u - e_i + e_j ∈ S
# ============================================================

def is_M_convex_support(support):
    S = set(support)
    if not S:
        return True
    k = len(next(iter(S)))
    for u in S:
        for v in S:
            if u == v:
                continue
            for i in range(k):
                if u[i] > v[i]:
                    # need j with u_j < v_j and u - e_i + e_j ∈ S
                    found = False
                    for j in range(k):
                        if u[j] < v[j]:
                            w = list(u); w[i] -= 1; w[j] += 1
                            if tuple(w) in S:
                                found = True
                                break
                    if not found:
                        return False
    return True

# ============================================================
# Lorentzian test for a homogeneous polynomial slice:
# For all |β|=m-2, g=∂^β f is quadratic; Hessian(g) must have ≤ 1 positive eigenvalue.
# ============================================================

def compositions(total, k):
    """All k-part compositions of 'total' into nonnegative integers."""
    if k == 1:
        yield (total,)
        return
    # recursive
    for first in range(total + 1):
        for rest in compositions(total - first, k - 1):
            yield (first,) + rest

def quadratic_hessian_matrix(g, var_names=None):
    """
    Build the symmetric Hessian H for a homogeneous quadratic g in QQ[y].
    H_ii = 2 * coeff(y_i^2),  H_ij = coeff(y_i y_j) for i≠j.
    """
    R = g.parent()
    y = R.gens()
    k = len(y)
    H = zero_matrix(QQ, k, k)
    for mon, coeff in g.dict().items():
        deg = sum(mon)
        if deg != 2:
            continue  # should not happen for homogeneous quadratic
        idx = [i for i,e in enumerate(mon) if e]
        if len(idx) == 1:
            i = idx[0]
            if mon[i] == 2:
                H[i,i] += 2 * coeff
        elif len(idx) == 2:
            i, j = idx
            # mon has exponents 1,1 at i,j
            H[i,j] += coeff
            H[j,i] += coeff
    return H

def lorentzian_slice_test(f, verbose=False):
    """
    Returns (is_lorentzian, failures) where failures is a list of
    (beta, eigenvalues) if any derivative fails the '≤1 positive eigenvalue' test.
    """
    R = f.parent()
    y = R.gens()
    k = len(y)
    m = f.total_degree()
    if m <= 1:
        return True, []  # linear/constant slices are trivially Lorentzian

    failures = []
    # iterate all β with |β|=m-2
    for beta in compositions(m-2, k):
        g = f
        for i, bi in enumerate(beta):
            if bi:
                for _ in range(bi):
                    g = g.derivative(y[i])
        # Must now be homogeneous quadratic (or zero)
        if g.is_zero():
            continue  # acceptable
        H = quadratic_hessian_matrix(g)
        # Work over RR to count positive eigenvalues robustly
        ev = [RR(x) for x in H.change_ring(RR).eigenvalues()]
        pos = sum(1 for lam in ev if lam > 1e-12)
        if pos > 1:
            failures.append((beta, ev))
            if verbose:
                print(f"Fail at beta={beta} with eigenvalues {ev}")
    return (len(failures) == 0), failures

# ============================================================
# End-to-end experiment driver
# ============================================================

def lorentzian_slices_experiment(polys, use_pick_2d=True, check_Mconvex=True, verbose=False):
    """
    For base polytopes polys=[P1,...,Pk], compute the multivariate Ehrhart
    in binomial basis up to total degree D=dim(sum P_i), then test for each m
    whether the slice f_m is Lorentzian. Optionally check M-convex support.
    Returns a dict with results and (if any) counterexamples.
    """
    k = len(polys)
    D = dimension_of_total(polys)  # maximal total degree
    counts = precompute_counts_grid(polys, D, use_pick_2d=use_pick_2d)
    c = coeffs_binomial_basis_from_counts(counts, D, k)

    results = {"degree": D, "per_m": []}
    var_names = [f"y{i+1}" for i in range(k)]

    for m in range(D + 1):
        f = polynomial_slice(c, m, k, var_names=var_names)
        supp = [alpha for alpha,val in c.items() if ell1(alpha)==m and val>0]
        mconv = True if not check_Mconvex else is_M_convex_support(supp)
        lorentz, fails = lorentzian_slice_test(f, verbose=verbose)
        results["per_m"].append({
            "m": m,
            "support_size": len(supp),
            "Mconvex": mconv,
            "Lorentzian": lorentz,
            "failures": fails,      # list of (beta, eigenvalues) if any
            "polynomial": f
        })
    return results

# ============================================================
# Example usage
# ============================================================
if __name__ == "__main__":
    # Simple 2D examples (Pick’s theorem gives big speedup)
    square  = Polyhedron(vertices=[(0,0),(1,0),(1,1),(0,1)])
    tri     = Polyhedron(vertices=[(0,0),(2,0),(0,1)])
    rect    = Polyhedron(vertices=[(0,0),(2,0),(2,1),(0,1)])

    examples = [
        [square, tri],
        [square, rect],
        [tri, rect],
    ]

    for idx, polys in enumerate(examples, 1):
        print(f"\n== Example {idx}: k={len(polys)} ==")
        report = lorentzian_slices_experiment(polys, use_pick_2d=True, check_Mconvex=True, verbose=False)
        print(f"Total degree D = {report['degree']}")
        for row in report["per_m"]:
            m  = row["m"]
            lc = "✓" if row["Lorentzian"] else "✗"
            mc = "✓" if row["Mconvex"]    else "✗"
            print(f"  m={m}:  support={row['support_size']:>2}  M-convex={mc}  Lorentzian={lc}  fails={len(row['failures'])}")
        # If you want to inspect a failing derivative:
        # print(report["per_m"][m]['polynomial'])
        # print(report["per_m"][m]['failures'][:3])

    # Random 2D trial
    rng = random.Random()
    def rand_poly(d=3, m=5, box=4):
        pts = set()
        while len(pts) < m:
            pts.add((rng.randint(-box, box), rng.randint(-box, box), rng.randint(-box, box)))
        P = Polyhedron(vertices=list(pts))
        if not is_bounded_poly(P) or P.dimension() < d:
            return rand_poly(d,m,box)
        return P

    P1, P2 = rand_poly(), rand_poly()
    print("\n== Random pair in 2D ==")
    rep = lorentzian_slices_experiment([P1, P2], use_pick_2d=True, check_Mconvex=True)
    for row in rep["per_m"]:
        print(f"m={row['m']}: Lorentzian={row['Lorentzian']}, Mconvex={row['Mconvex']}, support={row['support_size']}")



== Example 1: k=2 ==
Total degree D = 2
  m=0:  support= 1  M-convex=✓  Lorentzian=✓  fails=0
  m=1:  support= 2  M-convex=✓  Lorentzian=✓  fails=0
  m=2:  support= 3  M-convex=✓  Lorentzian=✓  fails=0

== Example 2: k=2 ==
Total degree D = 2
  m=0:  support= 1  M-convex=✓  Lorentzian=✓  fails=0
  m=1:  support= 2  M-convex=✓  Lorentzian=✓  fails=0
  m=2:  support= 3  M-convex=✓  Lorentzian=✓  fails=0

== Example 3: k=2 ==
Total degree D = 2
  m=0:  support= 1  M-convex=✓  Lorentzian=✓  fails=0
  m=1:  support= 2  M-convex=✓  Lorentzian=✓  fails=0
  m=2:  support= 3  M-convex=✓  Lorentzian=✓  fails=0

== Random pair in 2D ==


/var/folders/2h/x050p2112dng2w40dh8x3ltr0000gn/T/ipykernel_65663/855750719.py:277: UserWarning: Using generic algorithm for an inexact ring, which will probably give incorrect results due to numerical precision issues.
  ev = [RR(x) for x in H.change_ring(RR).eigenvalues()]


m=0: Lorentzian=True, Mconvex=True, support=1
m=1: Lorentzian=True, Mconvex=True, support=2
m=2: Lorentzian=True, Mconvex=True, support=3
m=3: Lorentzian=True, Mconvex=True, support=4


In [5]:
# SageMath ≥ 8.x

from sage.all import Polyhedron, ZZ
from itertools import product
#from sage.functions.binomial import binomial

# ------------------------------------------------------------
# Boundedness / counting (with a 2D Pick's theorem fast path)
# ------------------------------------------------------------

def is_bounded_poly(P):
    try:
        return P.is_compact()
    except AttributeError:
        return (len(P.rays_list()) == 0 and len(P.lines_list()) == 0)

def boundary_lattice_points_2d(P):
    """
    B(P) = sum over edges e of gcd(|dx_e|, |dy_e|)
    Works for lattice polygons.
    """
    B = ZZ(0)
    for e in P.faces(1):
        vs = [tuple(ZZ(c) for c in v.vector()) for v in e.vertices()]
        (x1, y1), (x2, y2) = vs
        dx = abs(x2 - x1); dy = abs(y2 - y1)
        # gcd in Sage handles gcd(n,0) correctly
        B += ZZ(dx).gcd(ZZ(dy))
    return B

def integral_points_count_fast(P, use_pick_2d=True, fallback_list_threshold=50000):
    """
    Count lattice points in P, preferring a fast exact method.
    - 2D: Pick's theorem (exact).
    - Else: Polyhedron.integral_points_count() if available.
    - Else: enumerate (guarded).
    """
    if P.is_empty():
        return 0

    # 2D fast path
    if use_pick_2d and P.ambient_dim() == 2:
        # Pick: |P∩Z^2| = Area(P) + B/2 + 1
        A = P.volume()                    # Euclidean area (exact rational)
        B = boundary_lattice_points_2d(P) # integer
        # Ensure exact integrality: 2*A is an integer for lattice polygons
        I2 = ZZ(2*A + B + 2)              # 2*|P∩Z^2|
        return I2 // 2

    # Generic fast path via Sage backends (Barvinok/Normaliz if configured)
    try:
        return P.integral_points_count()
    except Exception:
        pts = P.integral_points()
        if len(pts) > fallback_list_threshold:
            raise RuntimeError("Too many integer points to enumerate safely.")
        return len(pts)

# ------------------------------------------------------------
# Multiplicity-vector engine for AF computation
# ------------------------------------------------------------

def _minkowski_sum_scaled(polys, mult, scaled_cache, sum_cache):
    """
    Build Σ_i mult[i] * P_i as Minkowski sum of the *scaled* polytopes.
    Uses caches:
      - scaled_cache[(i,c)] = c * P_i
      - sum_cache[tuple(mult)] = Minkowski sum of the scaled parts
    """
    key = tuple(mult)
    if key in sum_cache:
        return sum_cache[key]

    parts = []
    for i, c in enumerate(mult):
        if c == 0:
            continue
        if c == 1:
            parts.append(polys[i])
        else:
            skey = (i, c)
            Q = scaled_cache.get(skey)
            if Q is None:
                Q = c*polys[i]
                scaled_cache[skey] = Q
            parts.append(Q)

    if not parts:
        # Σ_i 0*P_i = {0}; we won't construct it, caller handles count=1.
        sum_cache[key] = None
        return None

    S = parts[0]
    for Q in parts[1:]:
        S = S.minkowski_sum(Q)
    sum_cache[key] = S
    return S

def precompute_counts_by_mult(polys, max_mult, use_pick_2d=True):
    """
    Precompute #((Σ_i t_i P_i) ∩ Z^d) for all multiplicity vectors
    0 ≤ t_i ≤ max_mult[i]. Returns dict: mult(tuple) -> integer count.
    """
    d = len(polys)
    counts = {}
    scaled_cache = {}
    sum_cache = {}

    # sanity: inputs are polytopes
    #for P in polys:
    #    if not is_bounded_poly(P):
    #        raise ValueError("All inputs must be bounded (polytopes).")

    # Iterate all multiplicity combinations
    ranges = [range(m + 1) for m in max_mult]
    for mult in product(*ranges):
        if all(c == 0 for c in mult):
            counts[mult] = 1  # {0}
            continue
        S = _minkowski_sum_scaled(polys, mult, scaled_cache, sum_cache)
        #if not is_bounded_poly(S):
        #    raise ValueError("Minkowski sum became unbounded; check inputs.")
        counts[mult] = integral_points_count_fast(S, use_pick_2d=use_pick_2d)
    return counts

def cmv_from_multiplicities(counts, m):
    """
    CMV for a *multiset* of polytopes described by multiplicities m=(m1,...,md),
    where the underlying base polytopes are the keys of `counts`.

    Formula (inclusion–exclusion with duplicates):
      CMV = sum_{0 ≤ t_i ≤ m_i} [ (-1)^{k - Σ t_i} * (∏ binom(m_i, t_i)) * count(t) ],
    with k = Σ m_i and count(0,...,0)=1 for {0}.
    """
    k = sum(m)
    ranges = [range(mi + 1) for mi in m]
    total = ZZ(0)
    for t in product(*ranges):
        coeff = ZZ(1)
        s = 0
        for mi, ti in zip(m, t):
            coeff *= binomial(mi, ti)
            s += ti
        sign = -1 if ((k - s) % 2) else 1
        total += coeff * sign * ZZ(counts[t])
    return total

# ------------------------------------------------------------
# AF-type inequality with shared precomputation
# ------------------------------------------------------------

def af_cmv_values_optimized(polys, use_pick_2d=True):
    """
    Given base polytopes polys=[P1,...,Pd], compute:
      L  = CMV({1,1,1,...})          on [P1,P2,...,Pd]
      R1 = CMV({2,0,1,1,...})        duplicate P1, drop P2
      R2 = CMV({0,2,1,1,...})        duplicate P2, drop P1
    All three reuse one precomputed count table up to multiplicities (2,2,1,1,...).
    Returns (L, R1, R2, holds_boolean).
    """
    d = len(polys)
    if d < 2:
        raise ValueError("Need at least two polytopes.")

    # Correct multiplicity vectors
    mL  = tuple(1 for _ in range(d))
    mR1 = (2, 0) + tuple(1 for _ in range(d - 2))  # FIXED: zero out P2
    mR2 = (0, 2) + tuple(1 for _ in range(d - 2))  # FIXED: zero out P1

    # Sup multiplicities to cover all three CMVs:
    # allow up to 2 on both first two slots (for L, R1, R2); others need at most 1
    max_mult = (2, 2) + tuple(1 for _ in range(d - 2))

    counts = precompute_counts_by_mult(polys, max_mult, use_pick_2d=use_pick_2d)

    L  = cmv_from_multiplicities(counts, mL)
    R1 = cmv_from_multiplicities(counts, mR1)
    R2 = cmv_from_multiplicities(counts, mR2)
    return L, R1, R2, (L * L >= R1 * R2)


# ------------------------------------------------------------
# (Optional) plain CMV for an arbitrary list (no sharing)
# ------------------------------------------------------------

def combinatorial_mixed_valuation(polys, use_pick_2d=True):
    """
    CMV(P1,...,Pk) = sum_{I⊆[k]} (-1)^{k-|I|} |(Σ_{i∈I} P_i) ∩ Z^d|,
    with P_∅ = {0} and |{0}| = 1. Includes empty-set term.
    Generic version (no multiplicity sharing).
    """
    k = len(polys)
    if k == 0:
        return ZZ(1)

    # Build with subset DP (fresh caches)
    from itertools import combinations
    sum_cache = {}
    count_cache = {}
    indices = range(k)

    total = ZZ((-1)**k * 1)  # empty set term

    # Preload singletons
    for i, P in enumerate(polys):
        #if not is_bounded_poly(P):
        #    raise ValueError("Inputs must be polytopes (bounded).")
        sum_cache[frozenset([i])] = P

    for r in range(1, k+1):
        for I_tuple in combinations(indices, r):
            I = frozenset(I_tuple)
            if I not in sum_cache:
                j = next(iter(I))
                J = I - {j}
                sum_cache[I] = sum_cache[J].minkowski_sum(polys[j])
            S = sum_cache[I]
            if I not in count_cache:
                #if not is_bounded_poly(S):
                #    raise ValueError("Minkowski sum became unbounded.")
                count_cache[I] = integral_points_count_fast(S, use_pick_2d=use_pick_2d)
            total += ((-1) ** (k - r)) * ZZ(count_cache[I])

    return total

# ------------------------------------------------------------
# Quick demo / usage
# ------------------------------------------------------------
if __name__ == "__main__":
    # Some 2D examples (Pick's theorem will kick in automatically)
    square  = Polyhedron(vertices=[(0,0),(1,0),(1,1),(0,1)])
    tri     = Polyhedron(vertices=[(0,0),(2,0),(0,1)])
    rect    = Polyhedron(vertices=[(0,0),(2,0),(2,1),(0,1)])
    hex6    = Polyhedron(vertices=[(0,0),(1,0),(2,1),(2,2),(1,2),(0,1)])

    cube1 = Polyhedron

    print("== 2D canonical examples with AF test (optimized) ==")
    for P1, P2 in [(square, tri), (square, rect), (tri, rect), (square, hex6), (tri, hex6)]:
        L, R1, R2, ok = af_cmv_values_optimized([P1, P2], use_pick_2d=True)
        print(f"AF? {ok} | L^2={L*L},  R1*R2={R1*R2}")

    # Random trials (2D fast; 3D uses backend / enumeration)
    import random

    def random_full_dim_lattice_polytope(d, m=5, box=2, rng=None, max_tries=200):
        if rng is None:
            rng = random
        for _ in range(max_tries):
            pts = set()
            while len(pts) < m:
                pts.add(tuple(rng.randint(-box, box) for _ in range(d)))
            P = Polyhedron(vertices=list(pts))
            if P.dimension() == d:
                return P
        raise RuntimeError("Failed to generate a full-dimensional random lattice polytope.")

#    print("\n== Random trials in 2D ==")
#    rng = random.Random()
#    for _ in range(5):
#        P1 = random_full_dim_lattice_polytope(2, m=6, box=3, rng=rng)
#        P2 = random_full_dim_lattice_polytope(2, m=6, box=3, rng=rng)
#        L, R1, R2, ok = af_cmv_values_optimized([P1, P2], use_pick_2d=True)
#        print(f"AF? {ok} | L^2={L*L}, R1*R2={R1*R2}")

#    print("\n== Random trials in 3D ==")
    rng = random.Random()
    for _ in range(3):
        P1 = random_full_dim_lattice_polytope(5, m=7, box=2, rng=rng)
        P2 = random_full_dim_lattice_polytope(5, m=6, box=1, rng=rng)
        #P3 = random_full_dim_lattice_polytope(5, m=6, box=1, rng=rng)
        L, R1, R2, ok = af_cmv_values_optimized([P1, P2], use_pick_2d=True)
        print(f"AF? {ok} | L^2={L*L}, R1*R2={R1*R2}")

== 2D canonical examples with AF test (optimized) ==
AF? True | L^2=9,  R1*R2=4
AF? True | L^2=9,  R1*R2=8
AF? True | L^2=16,  R1*R2=8
AF? True | L^2=16,  R1*R2=12
AF? True | L^2=36,  R1*R2=12
AF? True | L^2=29929, R1*R2=1995
AF? True | L^2=113569, R1*R2=3258
AF? True | L^2=38809, R1*R2=1558


In [33]:
import random
k = random.randint(3, 15)
t = random.randint(3, 15)
A = matrix(QQ,[[(1/2)*t^2, 4*t*k, binomial(t+2, 2)-1],[2*t*k, k^2, (k+1)^2-1], [binomial(t+2,2)-1, (k+1)^2-1, 1]])

In [34]:
A.eigenvalues()

[-310.8930756413172?, -40.24294264574292?, 617.636018287061?]

In [37]:
#Vertices lists: [[(-2, -2), (-1, -2), (3, 3), (-1, 1)], [(-3, 0), (-3, 1), (1, -3), (-1, 0)], [(-3, -3), (-1, -3), (-1, 1), (2, -2), (3, 2)]]

In [41]:
P = Polyhedron(vertices=[(1, -2, -1), (1, -2, 1), (1, 0, -1), (2, 0, -3), (3, -1, 3), (3, 1, 2)])
Q = Polyhedron(vertices=[(-3, 3, -1), (-2, 1, -2), (-1, 3, -3), (0, 3, -3), (2, -3, 3), (3, -1, -1)])
R = Polyhedron(vertices=[(-3, 0, -1), (0, 1, 3), (2, -3, 2), (2, 0, -2), (2, 1, 1), (2, 2, -2)])

In [43]:
integral_points_count_fast(P)

15

In [ ]:
P = Polyhedron(vertices=[(1, -2, -1), (1, -2, 1), (1, 0, -1), (2, 0, -3), (3, -1, 3), (3, 1, 2)])
Q = Polyhedron(vertices=[(-3, 3, -1), (-2, 1, -2), (-1, 3, -3), (0, 3, -3), (2, -3, 3), (3, -1, -1)])
R = Polyhedron(vertices=[(-3, 0, -1), (0, 1, 3), (2, -3, 2), (2, 0, -2), (2, 1, 1), (2, 2, -2)])

In [45]:
# ====== Paste your vertices here (integer coordinates) ======
V1 = [ (1, -2, -1), (1, -2, 1), (1, 0, -1), (2, 0, -3), (3, -1, 3), (3, 1, 2) ]   # <-- replace
V2 = [ (-3, 3, -1), (-2, 1, -2), (-1, 3, -3), (0, 3, -3), (2, -3, 3), (3, -1, -1) ]   # <-- replace
V3 = [ (-3, 0, -1), (0, 1, 3), (2, -3, 2), (2, 0, -2), (2, 1, 1), (2, 2, -2) ]   # <-- replace
# ============================================================

from sage.all import (
    Polyhedron, QQ, ZZ, RR, PolynomialRing,
    factorial, binomial, zero_matrix
)
from itertools import product

# ---------- helpers (portable; no .scale; safe Hessian) ----------
def is_bounded_poly(P):
    try:
        return P.is_compact()
    except AttributeError:
        return (len(P.rays_list()) == 0 and len(P.lines_list()) == 0)

def integral_points_count_fast(P, fallback_list_threshold=200000):
    if P.is_empty():
        return 0
    try:
        return P.integral_points_count()
    except Exception:
        pts = P.integral_points()
        if len(pts) > fallback_list_threshold:
            raise RuntimeError("Too many integer points to enumerate safely.")
        return len(pts)

def scale_polyhedron(P, c):
    if c == 0:
        return None
    if c == 1:
        return P
    lam = QQ(c)
    verts = [tuple(lam * v.vector()) for v in P.vertices()]
    return Polyhedron(vertices=verts)

def _minkowski_sum_scaled(polys, mult, scaled_cache, sum_cache):
    key = tuple(mult)
    if key in sum_cache:
        return sum_cache[key]
    parts = []
    for i, c in enumerate(mult):
        if c == 0: continue
        if c == 1:
            parts.append(polys[i])
        else:
            skey = (i, c)
            Q = scaled_cache.get(skey)
            if Q is None:
                Q = scale_polyhedron(polys[i], c)
                scaled_cache[skey] = Q
            parts.append(Q)
    if not parts:
        sum_cache[key] = None
        return None
    S = parts[0]
    for Q in parts[1:]:
        S = S.minkowski_sum(Q)
    sum_cache[key] = S
    return S

def dimension_of_total(polys):
    S = polys[0]
    for Q in polys[1:]:
        S = S.minkowski_sum(Q)
    return S.dimension()

def precompute_counts_grid(polys, D):
    for P in polys:
        if not is_bounded_poly(P):
            raise ValueError("All inputs must be bounded polytopes.")
    k = len(polys)
    ranges = [range(D+1) for _ in range(k)]
    counts = {}
    scaled_cache, sum_cache = {}, {}
    for t in product(*ranges):
        if all(c == 0 for c in t):
            counts[t] = ZZ(1)  # {0}
            continue
        S = _minkowski_sum_scaled(polys, t, scaled_cache, sum_cache)
        if not is_bounded_poly(S):
            raise ValueError("Minkowski sum became unbounded; check inputs.")
        counts[t] = ZZ(integral_points_count_fast(S))
    return counts

def coeffs_binomial_basis_from_counts(counts, D, k):
    def L1(a): return sum(a)
    c = {}
    for alpha in product(*[range(D+1) for _ in range(k)]):
        if L1(alpha) > D: continue
        total = ZZ(0)
        for s in product(*[range(ai+1) for ai in alpha]):
            sign = -1 if ((L1(alpha) - sum(s)) % 2) else 1
            w = ZZ(1)
            for ai, si in zip(alpha, s):
                w *= binomial(ai, si)
            total += sign * w * counts[s]
        c[alpha] = total
    return c

def homogenized_polynomial(c, D, k, var_names=None):
    #  \hat f(y,z) = sum_alpha c_alpha / (alpha! (D-|alpha|)!) y^alpha z^{D-|alpha|}
    if var_names is None:
        var_names = [f"y{i+1}" for i in range(k)] + ["z"]
    R = PolynomialRing(QQ, k+1, names=var_names)
    y = R.gens()[:-1]; z = R.gens()[-1]
    f = R.zero()
    for alpha, val in c.items():
        deg = sum(alpha)
        if deg > D or val == 0: continue
        mon = R(1)
        for i, ai in enumerate(alpha):
            if ai: mon *= y[i]**ai
        mon *= z**(D - deg)
        alpha_fact = 1
        for ai in alpha: alpha_fact *= factorial(ai)
        f += QQ(val) / (QQ(alpha_fact) * QQ(factorial(D-deg))) * mon
    return f

def compositions(total, k):
    if k == 1:
        yield (total,); return
    for first in range(total+1):
        for rest in compositions(total-first, k-1):
            yield (first,) + rest

def quadratic_hessian_matrix(g):
    R = g.parent(); vars_ = R.gens(); k = len(vars_)
    H = zero_matrix(QQ, k, k)
    zero_subs = {v: 0 for v in vars_}
    for i in range(k):
        for j in range(k):
            sec = g.derivative(vars_[i]).derivative(vars_[j])
            H[i,j] = QQ(sec.subs(zero_subs))  # constants for quadratics
    return H

def lorentzian_homogeneous_test(f, tol=1e-13, verbose=True):
    R = f.parent(); vars_ = R.gens(); k = len(vars_)
    m = f.total_degree()
    if m <= 1: return True, []
    # Require nonnegative coefficients (typical for our normalization)
    for _, coeff in f.dict().items():
        if coeff < 0:
            if verbose: print("Negative coefficient encountered.")
            return False, [("negative_coeff", None)]
    fails = []
    for beta in compositions(m-2, k):
        g = f
        for i, bi in enumerate(beta):
            for _ in range(bi):
                g = g.derivative(vars_[i])
        if g.is_zero():  # allowed
            continue
        H = quadratic_hessian_matrix(g)
        ev = [RR(x) for x in H.change_ring(RR).eigenvalues()]
        pos = sum(1 for lam in ev if lam > tol)
        if pos > 1:
            fails.append((beta, ev))
            if verbose:
                print(f"FAIL beta={beta}, eigenvalues={ev}")
    return (len(fails) == 0), fails

# ---------- build polytopes and verify ----------
P1, P2, P3 = Polyhedron(vertices=V1), Polyhedron(vertices=V2), Polyhedron(vertices=V3)
for i, P in enumerate([P1,P2,P3], 1):
    if not is_bounded_poly(P):
        raise ValueError(f"P{i} is not bounded.")
    print(f"P{i}: dim={P.dimension()}, #verts={len(P.vertices())}")

polys = [P1,P2,P3]
D = dimension_of_total(polys)
print(f"Total degree D (dim of P1+P2+P3) = {D}")

counts = precompute_counts_grid(polys, D)
c = coeffs_binomial_basis_from_counts(counts, D, k=3)
Fhat = homogenized_polynomial(c, D, k=3)

print("\nHomogenized polynomial (degree D) built over QQ in variables", Fhat.parent().gens())
ok, fails = lorentzian_homogeneous_test(Fhat, tol=1e-12, verbose=True)
print("\nLorentzian (homogenized) =", ok)
if not ok:
    print("Number of failing derivatives:", len(fails))
    print("First 3 failures (beta, eigenvalues):", fails[:3])

# (Optional) also show slice results, esp. m=2
def slice_poly(m):
    R = PolynomialRing(QQ, 3, names=("y1","y2","y3")); y = R.gens()
    f = R.zero()
    for alpha, val in c.items():
        if sum(alpha) != m or val == 0: continue
        mon = R(1)
        for i, ai in enumerate(alpha):
            if ai: mon *= y[i]**ai
        # normalize by alpha!
        alpha_fact = 1
        for ai in alpha: alpha_fact *= factorial(ai)
        f += QQ(val)/QQ(alpha_fact) * mon
    return f

print("\nSlice m=2 check (should be Lorentzian by mixed volumes):")
f2 = slice_poly(2)
ok2, fails2 = lorentzian_homogeneous_test(f2, tol=1e-12, verbose=False)
print("Lorentzian (m=2 slice) =", ok2)
if not ok2:
    print("m=2 failures:", fails2)


P1: dim=3, #verts=6
P2: dim=3, #verts=6
P3: dim=3, #verts=6
Total degree D (dim of P1+P2+P3) = 3

Homogenized polynomial (degree D) built over QQ in variables (y1, y2, y3, z)
FAIL beta=(0, 0, 0, 1), eigenvalues=[608.853307457101, 0.539271982484664, -81.2506073854328, -182.141972054153]

Lorentzian (homogenized) = False
Number of failing derivatives: 1
First 3 failures (beta, eigenvalues): [((0, 0, 0, 1), [608.853307457101, 0.539271982484664, -81.2506073854328, -182.141972054153])]

Slice m=2 check (should be Lorentzian by mixed volumes):
Lorentzian (m=2 slice) = True


/var/folders/2h/x050p2112dng2w40dh8x3ltr0000gn/T/ipykernel_65663/1438949738.py:158: UserWarning: Using generic algorithm for an inexact ring, which will probably give incorrect results due to numerical precision issues.
  ev = [RR(x) for x in H.change_ring(RR).eigenvalues()]


In [4]:
# === Degree-2 Lorentzian tester for the binomial-basis mixed Ehrhart polynomial ===
# Works for any k ≥ 2 polytopes in Z^d. Minimal counting (O(k^2) Minkowski sums).
# Uses a 2D Pick’s theorem fast path; portable scaling; mutable Hessian.

from sage.all import (
    Polyhedron, QQ, ZZ, RR, PolynomialRing,
    zero_matrix
)

# ----------------- Helpers: boundedness, counting, scaling, sums -----------------

def is_bounded_poly(P):
    try:
        return P.is_compact()
    except AttributeError:
        return (len(P.rays_list()) == 0 and len(P.lines_list()) == 0)

def boundary_lattice_points_2d(P):
    B = ZZ(0)
    for e in P.faces(1):
        (x1,y1), (x2,y2) = [tuple(ZZ(c) for c in v.vector()) for v in e.vertices()]
        dx, dy = abs(x2-x1), abs(y2-y1)
        B += ZZ(dx).gcd(ZZ(dy))
    return B

def integral_points_count_fast(P, use_pick_2d=True, fallback_list_threshold=200000):
    if P.is_empty():
        return 0
    if use_pick_2d and P.ambient_dim() == 2:
        # Pick's theorem: |P∩Z^2| = Area + B/2 + 1 (exact for lattice polygons)
        A = P.volume()                     # exact rational
        B = boundary_lattice_points_2d(P)  # integer
        return ZZ((2*A + B + 2) // 2)
    try:
        return P.integral_points_count()
    except Exception:
        pts = P.integral_points()
        if len(pts) > fallback_list_threshold:
            raise RuntimeError("Too many integer points to enumerate safely.")
        return len(pts)

def scale_polyhedron(P, c):
    """
    Return c*P by scaling vertices (portable). c is a nonnegative integer.
    """
    if c == 0:
        return None   # caller handles {0}
    if c == 1:
        return P
    lam = QQ(c)
    verts = [tuple(lam * v.vector()) for v in P.vertices()]
    return Polyhedron(vertices=verts)

def minkowski_sum_scaled_pair(Pi, Pj=None, si=1, sj=1):
    """
    Return si*Pi (+ sj*Pj if provided). si,sj nonnegative integers.
    """
    A = scale_polyhedron(Pi, si)
    if Pj is None:      # only one polytope
        return A
    B = scale_polyhedron(Pj, sj)
    if A is None:       # {0} + sj*Pj
        return B
    if B is None:       # si*Pi + {0}
        return A
    return A.minkowski_sum(B)

# ----------------- Degree-2 binomial coefficients and normalized quadratic -------

def degree2_binomial_coeffs(polys, use_pick_2d=True):
    """
    For k polytopes, compute the |α|=2 binomial-basis coefficients:
      c_{2e_i}   = F(2e_i) - 2 F(e_i) + F(0),
      c_{e_i+e_j}= F(e_i+e_j) - F(e_i) - F(e_j) + F(0)
    where F(t) = |(Σ t_i P_i) ∩ Z^d| and F(0)=1.
    Returns (diag, off) where:
      diag[i]       = c_{2e_i}
      off[(i,j)]    = c_{e_i+e_j} for i<j
    """
    k = len(polys)
    for P in polys:
        if not is_bounded_poly(P):
            raise ValueError("All inputs must be bounded polytopes.")

    F0 = ZZ(1)
    F1 = [ZZ(integral_points_count_fast(polys[i], use_pick_2d=use_pick_2d)) for i in range(k)]
    F2 = [ZZ(integral_points_count_fast(scale_polyhedron(polys[i], 2), use_pick_2d=use_pick_2d)) for i in range(k)]

    F11 = {}
    for i in range(k):
        for j in range(i+1, k):
            S = minkowski_sum_scaled_pair(polys[i], polys[j], si=1, sj=1)
            F11[(i,j)] = ZZ(integral_points_count_fast(S, use_pick_2d=use_pick_2d))

    diag = [F2[i] - 2*F1[i] + F0 for i in range(k)]
    off  = {(i,j): F11[(i,j)] - F1[i] - F1[j] + F0 for i in range(k) for j in range(i+1,k)}
    return diag, off

def degree2_normalized_quadratic(polys, var_names=None, use_pick_2d=True):
    """
    Build f2(y) = Σ_i (c_{2e_i}/2) y_i^2 + Σ_{i<j} c_{e_i+e_j} y_i y_j  over QQ,
    where c_α are |α|=2 binomial-basis coefficients.
    (Normalization by α! is implicit: α!=(2,0,...)->2 for diag; α!=(1,1,...)->1 off-diag.)
    """
    k = len(polys)
    if var_names is None:
        var_names = [f"y{i+1}" for i in range(k)]
    R = PolynomialRing(QQ, k, names=var_names); y = R.gens()

    diag, off = degree2_binomial_coeffs(polys, use_pick_2d=use_pick_2d)
    f = R.zero()
    # diagonal terms: divide by 2! = 2
    for i in range(k):
        if diag[i]:
            f += QQ(diag[i]) / QQ(2) * (y[i]**2)
    # off-diagonal terms: divide by 1!*1! = 1
    for (i,j), c in off.items():
        if c:
            f += QQ(c) * (y[i]*y[j])
    return f, diag, off

# ----------------- Lorentzian test for the quadratic (Hessian signature) ---------

def quadratic_hessian_matrix(g):
    """
    Hessian via second partials (constant for quadratics).
    For f2 = (1/2) x^T H x + ..., Sage’s H matches:
      H_ii = 2 * coeff(y_i^2),   H_ij = coeff(y_i y_j) for i≠j.
    """
    R = g.parent(); vars_ = R.gens(); k = len(vars_)
    H = zero_matrix(QQ, k, k)
    zero_subs = {v: 0 for v in vars_}
    for i in range(k):
        for j in range(k):
            sec = g.derivative(vars_[i]).derivative(vars_[j])
            H[i,j] = QQ(sec.subs(zero_subs))  # constants for quadratics
    return H

def degree2_lorentzian_test(polys, var_names=None, use_pick_2d=True, tol=1e-12, verbose=True):
    """
    Compute f2 (normalized) and test Lorentzianity:
    '≤ 1 positive eigenvalue' for the Hessian (equivalently for degree-2).
    Returns dict with f2, Hessian, eigenvalues, and boolean result.
    """
    f2, diag, off = degree2_normalized_quadratic(polys, var_names=var_names, use_pick_2d=use_pick_2d)
    H = quadratic_hessian_matrix(f2)
    ev = [RR(x) for x in H.change_ring(RR).eigenvalues()]
    pos = sum(1 for lam in ev if lam > tol)
    ok = (pos <= 1)
    if verbose:
        print("f2(y) =", f2)
        print("Hessian ="); print(H)
        print("Eigenvalues =", ev)
        print("Lorentzian (degree 2) =", ok)
    return {"f2": f2, "H": H, "eigenvalues": ev, "lorentzian": ok, "diag_c2ei": diag, "off_c_ei_ej": off}
def poly_from_vertices(verts):
    return Polyhedron(vertices=[vector(ZZ, v) for v in verts])
# ----------------- Example usage -----------------
if __name__ == "__main__":
    # Replace these with your polytopes (any k ≥ 2). Example in 3D with k=3:
    P1 = poly_from_vertices([(0, 0, 0), (1, 0, 0)])
    P2 = poly_from_vertices([(0, 0, 0), (0, 1, 0)])
    P3 = poly_from_vertices([(0, 0, 0), (0, 0, 1)])
    P4 = poly_from_vertices([(0, 0, 0), (0, 1, 0), (1, 0, 0), (1,1,0)])
    P5 = poly_from_vertices([(0, 0, 0), (0, 0, 1), (1, 0, 0),(1,0,1)])
    P6 = poly_from_vertices([(0, 0, 0), (0, 1, 0), (0, 0, 1),(0,1,1)])
    #P1, P2, P3 = Polyhedron(vertices=V1), Polyhedron(vertices=V2), Polyhedron(vertices=V3)

    res = degree2_lorentzian_test([P1,P2,P3,P4,P5,P6], use_pick_2d=False, verbose=True)


f2(y) = y1*y2 + y1*y3 + y2*y3 + y1*y4 + y2*y4 + 3*y3*y4 + y4^2 + y1*y5 + 3*y2*y5 + y3*y5 + 5*y4*y5 + y5^2 + 3*y1*y6 + y2*y6 + y3*y6 + 5*y4*y6 + 5*y5*y6 + y6^2
Hessian =
[0 1 1 1 1 3]
[1 0 1 1 3 1]
[1 1 0 3 1 1]
[1 1 3 2 5 5]
[1 3 1 5 2 5]
[3 1 1 5 5 2]
Eigenvalues = [14.0710678118655, 0.236067977499790, 0.236067977499790, -0.0710678118654752, -4.23606797749979, -4.23606797749979]
Lorentzian (degree 2) = False


/var/folders/2h/x050p2112dng2w40dh8x3ltr0000gn/T/ipykernel_12108/3538574211.py:147: UserWarning: Using generic algorithm for an inexact ring, which will probably give incorrect results due to numerical precision issues.
  ev = [RR(x) for x in H.change_ring(RR).eigenvalues()]


In [16]:
# ===================== CMV(Alexandrov–Fenchel) degree-2 counterexample hunter =====================
# SageMath ≥ 9.x

from sage.all import (
    Polyhedron, QQ, ZZ, RR, randint, binomial, cached_function
)
from itertools import product
import random

# -------------------------------- Basics: boundedness, counting, scaling --------------------------------

def is_bounded_poly(P):
    try:
        return P.is_compact()
    except AttributeError:
        return (len(P.rays_list()) == 0 and len(P.lines_list()) == 0)

def boundary_lattice_points_2d(P):
    B = ZZ(0)
    for e in P.faces(1):
        (x1,y1), (x2,y2) = [tuple(ZZ(c) for c in v.vector()) for v in e.vertices()]
        dx, dy = abs(x2-x1), abs(y2-y1)
        B += ZZ(dx).gcd(ZZ(dy))
    return B

def integral_points_count_fast(P, use_pick_2d=True, fallback_list_threshold=500000):
    """Count |P ∩ Z^d| with a 2D exact fast path, else backend, else guarded enumeration."""
    if P is None:
        return 1  # treat {0}
    if P.is_empty():
        return 0
    d = P.ambient_dim()
    if use_pick_2d and d == 2:
        A = P.volume()                     # exact area (QQ)
        B = boundary_lattice_points_2d(P)  # integer
        return ZZ((2*A + B + 2) // 2)      # Pick's theorem
    try:
        return ZZ(P.integral_points_count())
    except Exception:
        pts = P.integral_points()
        if len(pts) > fallback_list_threshold:
            raise RuntimeError("Too many integer points to enumerate safely.")
        return ZZ(len(pts))

def scale_polyhedron(P, c):
    """Return c*P (c ∈ Z_{≥0}) by scaling vertices. Portable (no .scale)."""
    if c == 0:
        return None     # represent {0}; caller handles count=1
    if c == 1:
        return P
    lam = QQ(c)
    verts = [tuple(lam * v.vector()) for v in P.vertices()]
    return Polyhedron(vertices=verts)

# -------------------------------- CMV degree-2 ingredients for a pair (P,Q) -------------------------------

def cmv_degree2_values(P, Q, use_pick_2d=True):
    """
    Returns (L, R1, R2) where
       L  = E(P+Q) - E(P) - E(Q) + 1
       R1 = E(2P)  - 2E(P) + 1
       R2 = E(2Q)  - 2E(Q) + 1
    """
    if not (is_bounded_poly(P) and is_bounded_poly(Q)):
        raise ValueError("Inputs must be bounded polytopes.")

    EP  = integral_points_count_fast(P, use_pick_2d=use_pick_2d)
    EQ  = integral_points_count_fast(Q, use_pick_2d=use_pick_2d)
    E2P = integral_points_count_fast(scale_polyhedron(P, 2), use_pick_2d=use_pick_2d)
    E2Q = integral_points_count_fast(scale_polyhedron(Q, 2), use_pick_2d=use_pick_2d)
    S   = P.minkowski_sum(Q)
    EPQ = integral_points_count_fast(S, use_pick_2d=use_pick_2d)

    L  = EPQ - EP - EQ + 1
    R1 = E2P - 2*EP + 1
    R2 = E2Q - 2*EQ + 1
    return ZZ(L), ZZ(R1), ZZ(R2)

def violates_cmv_degree2(P, Q, use_pick_2d=True):
    """Return (violates_bool, values_dict). Violation if L^2 < R1*R2."""
    L, R1, R2 = cmv_degree2_values(P, Q, use_pick_2d=use_pick_2d)
    left  = L*L
    right = R1*R2
    return (left < right), {"L": L, "R1": R1, "R2": R2, "L2": left, "R1R2": right}

# -------------------------------- Polytope factories (families likely to stress inequality) ---------------

def rand_lattice_points(d, m, box, rng):
    pts = set()
    while len(pts) < m:
        pts.add(tuple(rng.randint(-box, box) for _ in range(d)))
    return list(pts)

def random_full_dim_polytope(d, m=8, box=4, rng=None, max_tries=400):
    """Random convex hull of m lattice points in [-box,box]^d, full-dim & bounded."""
    if rng is None: rng = random
    for _ in range(max_tries):
        P = Polyhedron(vertices=rand_lattice_points(d, m, box, rng))
        if is_bounded_poly(P) and P.dimension() == d:
            return P
    raise RuntimeError("Failed to generate a full-dimensional random polytope.")

def random_zonotope(d, s=8, box=3, rng=None):
    """
    Zonotope: Minkowski sum of segments [0, v_i].
    Return its vertex hull (convex hull of {sum λ_i v_i | λ_i∈{0,1}}).
    """
    if rng is None: rng = random
    gens = [tuple(rng.randint(-box, box) for _ in range(d)) for _ in range(s)]
    # generate all subset sums
    vs = [tuple(0 for _ in range(d))]
    for v in gens:
        vs += [tuple(a+b for a,b in zip(u,v)) for u in vs]
    P = Polyhedron(vertices=vs)
    # center to reduce coordinates (optional)
    return P

def product_of_segments(lengths):
    """Return the axis-aligned box ∏ [0, L_i]."""
    d = len(lengths)
    verts = []
    for bits in product([0,1], repeat=d):
        verts.append(tuple(bits[i]*lengths[i] for i in range(d)))
    return Polyhedron(vertices=verts)

def reeve_tetrahedron(h):
    """Classical Reeve tetrahedron (non-IDP): conv{(0,0,0),(1,0,0),(0,1,0),(1,1,h)}."""
    V = [(0,0,0),(1,0,0),(0,1,0),(1,1,h)]
    return Polyhedron(vertices=V)

# -------------------------------- Search harness (serial or parallel) -------------------------------------

def random_pair_factory(d, rng, family="mixed"):
    """
    Make (P,Q) according to 'family':
      - 'random' : random hulls
      - 'zonotope': zonotopes
      - 'box'    : random axis-aligned boxes (products of segments)
      - 'reeve'  : mixes Reeve tetrahedra and random hulls (d must be 3)
      - 'mixed'  : cycles through several families
    """
    if family == "random":
        P = random_full_dim_polytope(d, m=d+2, box=2, rng=rng)
        Q = random_full_dim_polytope(d, m=d+1, box=2, rng=rng)
        return P, Q
    if family == "zonotope":
        s = 3
        return random_zonotope(d, s=s, box=3, rng=rng), random_zonotope(d, s=s, box=3, rng=rng)
    if family == "box":
        L1 = [rng.randint(1,6) for _ in range(d)]
        L2 = [rng.randint(1,6) for _ in range(d)]
        return product_of_segments(L1), product_of_segments(L2)
    if family == "reeve":
        if d != 3:
            raise ValueError("Reeve family requires d=3.")
        return reeve_tetrahedron(rng.randint(2,12)), random_full_dim_polytope(3, m=8, box=3, rng=rng)
    if family == "mixed":
        choice = rng.choice(["random","zonotope","box"] + (["reeve"] if d==3 else []))
        return random_pair_factory(d, rng, family=choice)
    raise ValueError("Unknown family.")

def search_cmv_degree2_counterexample(d=3, trials=200, seed=0, family="mixed", use_pick_2d=True, verbose=True):
    """
    Try 'trials' random pairs (P,Q) in dimension d. Return the first violation or None.
    """
    rng = random.Random(seed)
    for t in range(trials):
        P, Q = random_pair_factory(d, rng, family=family)
        bad, vals = violates_cmv_degree2(P, Q, use_pick_2d=use_pick_2d)
        if verbose and t % max(1, trials//10) == 0:
            print(f"[{t}/{trials}] L^2={vals['L2']}, R1R2={vals['R1R2']}  ({'VIOLATION' if bad else 'ok'})")
        if bad:
            return {
                "trial": t,
                "dimension": d,
                "family": family,
                "values": vals,
                "P_vertices": [tuple(v.vector()) for v in P.vertices()],
                "Q_vertices": [tuple(v.vector()) for v in Q.vertices()]
            }
    return None

# ------------------------------ Parallel wrapper (optional) ---------------------------------
# Usage: set PARALLEL=True, then run search_parallel(...)

PARALLEL = False  # set True to enable @parallel trials

if PARALLEL:
    @parallel
    def _trial_worker(args):
        d, seed, family, use_pick_2d = args
        rng = random.Random(seed)
        P, Q = random_pair_factory(d, rng, family=family)
        bad, vals = violates_cmv_degree2(P, Q, use_pick_2d=use_pick_2d)
        if bad:
            return True, {
                "dimension": d, "family": family, "values": vals,
                "P_vertices": [tuple(v.vector()) for v in P.vertices()],
                "Q_vertices": [tuple(v.vector()) for v in Q.vertices()]
            }
        return False, None

def search_parallel(d=3, trials=1000, seed=0, family="mixed", use_pick_2d=True):
    """
    Parallel search (set PARALLEL=True above). Returns first violation or None.
    """
    if not PARALLEL:
        raise RuntimeError("Enable PARALLEL=True to use search_parallel.")
    tasks = [ (d, seed+i, family, use_pick_2d) for i in range(trials) ]
    for (_, res) in _trial_worker(tasks):
        ok, payload = res
        if ok:
            return payload
    return None

# -------------------------------- Demo / CLI-ish entry ----------------------------------------------------

if __name__ == "__main__":
    # Quick test sweep (adjust as you like)
    print("=== CMV degree-2 counterexample search ===")
    # D=2 is extremely fast (Pick); D=3 pushes harder examples
    for d in [7]:
        print(f"\n-- dimension d={d} --")
        hit = search_cmv_degree2_counterexample(
            d=d, trials=400, seed=None, family="mixed",
            use_pick_2d=True, verbose=True
        )
        if hit is None:
            print("No counterexample found in this run.")
        else:
            print("\n*** COUNTEREXAMPLE FOUND ***")
            print("Trial:", hit["trial"])
            print("Dimension:", hit["dimension"])
            print("Family:", hit["family"])
            print("Values:", hit["values"])
            print("P vertices:", hit["P_vertices"])
            print("Q vertices:", hit["Q_vertices"])
            break

    # Example: brute-force Reeve vs random in 3D (often challenging)
    #print("\n-- Reeve family sweep (3D) --")
    #hit = search_cmv_degree2_counterexample(
    #    d=3, trials=300, seed=None, family="reeve",
    #    use_pick_2d=False, verbose=False
    #)
    #if hit:
    #    print("*** COUNTEREXAMPLE FOUND (Reeve mix) ***")
    #    print(hit)
    #else:
    #    print("No counterexample (Reeve mix) in this run.")


=== CMV degree-2 counterexample search ===

-- dimension d=7 --
[0/400] L^2=2401, R1R2=144  (ok)


KeyboardInterrupt: 